In [1]:
import pandas as pd
import numpy as np
import sqlite3
from geopy.geocoders import Nominatim

In [5]:
df = pd.read_csv('logiPrime/amazon_delivery.csv.zip')

In [6]:
df.head(5)

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [7]:
df.columns

Index(['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude',
       'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date',
       'Order_Time', 'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 'Area',
       'Delivery_Time', 'Category'],
      dtype='object')

In [8]:
df = df.rename(columns={
    'Order_ID': 'IdPedido',
    'Agent_Age': 'IddEntragador',
    'Agent_Rating': 'NtEntragador',
    'Store_Latitude': 'LatGalpao',
    'Store_Longitude': 'LonGalpao',
    'Drop_Latitude': 'LatDestino',
    'Drop_Longitude': 'LonDestino',
    'Order_Date': 'DtPedido',
    'Order_Time': 'Hrpedido',
    'Pickup_Time': 'HrColeta',
    'Weather': 'Clima',
    'Traffic': 'Transito',
    'Vehicle': 'Veiculo',
    'Area': 'Area',
    'Delivery_Time': 'TpEntrega',
    'Category': 'CtProduto'
})

In [9]:
df = df.rename(columns={
    'IddEntragador': 'IddEntregador',
    'NtEntragador': 'NtEntregador',
})

# Tabela de Agentes

In [10]:
df['IdEntregador'] = df.groupby(
    ['IddEntregador', 'NtEntregador', 'Veiculo']
).ngroup()

In [11]:
print(f'Número de Linhas: {df.shape[0]}\nNúmero de Entregadores: {df['IdEntregador'].nunique()}')

Número de Linhas: 43739
Número de Entregadores: 1017


In [12]:
df['categoria'] = pd.cut(
    df['NtEntregador'],
    bins=[-float('inf'), 3, 4, 4.5, float('inf')],
    labels=['Ruim', 'Regular', 'Bom', 'Excelente']
)

In [13]:
def gerar_nomes_entregadores(df, col_id='IdEntregador'):
    nomes_masculinos = [
        'João', 'Pedro', 'Lucas', 'Gabriel', 'Mateus',
        'Rafael', 'Bruno', 'Felipe', 'Carlos', 'André'
    ]

    nomes_femininos = [
        'Maria', 'Ana', 'Juliana', 'Fernanda', 'Patricia',
        'Camila', 'Amanda', 'Larissa', 'Beatriz', 'Carolina'
    ]

    sobrenomes = [
        'Silva', 'Santos', 'Oliveira', 'Souza', 'Rodrigues',
        'Ferreira', 'Alves', 'Pereira', 'Lima', 'Gomes',
        'Costa', 'Ribeiro', 'Martins', 'Carvalho', 'Almeida',
        'Lopes', 'Soares', 'Fernandes', 'Vieira', 'Barbosa',
        'Rocha', 'Dias', 'Monteiro', 'Teixeira', 'Moreira'
    ]

    ids_unicos = df[col_id].unique()

    mapa_nomes = {}
    mapa_genero = {}

    for _id in ids_unicos:
        genero = np.random.choice(['Masculino', 'Feminino'])

        if genero == 'Masculino':
            nome = np.random.choice(nomes_masculinos)
        else:
            nome = np.random.choice(nomes_femininos)

        sobrenome1, sobrenome2 = np.random.choice(sobrenomes, size=2, replace=False)

        nome_completo = f"{nome} {sobrenome1} {sobrenome2}"

        mapa_nomes[_id] = nome_completo
        mapa_genero[_id] = genero

    df['NmEntregador'] = df[col_id].map(mapa_nomes)
    df['Genero'] = df[col_id].map(mapa_genero)

    return df

In [14]:
df = gerar_nomes_entregadores(df, col_id='IdEntregador')

In [ ]:
df.columns

Index(['IdPedido', 'IddEntregador', 'NtEntregador', 'LatGalpao', 'LonGalpao',
       'LatDestino', 'LonDestino', 'DtPedido', 'Hrpedido', 'HrColeta', 'Clima',
       'Transito', 'Veiculo', 'Area', 'TpEntrega', 'CtProduto', 'IdEntregador',
       'categoria', 'NmEntregador', 'GrEntregador'],
      dtype='object')

In [15]:
df = df.rename(columns={
    'Genero': 'GrEntregador',
})

In [16]:
df = df.rename(columns={
    'categoria': 'CtEntregador',
})

In [17]:
df_entregadores = df[
    ['IdEntregador', 'IddEntregador', 'Veiculo', 'CtEntregador', 'NmEntregador', 'GrEntregador']
]

# Tabela De Galpões

In [18]:
df['IdGalpao'] = df.groupby(
    ['LatGalpao', 'LonGalpao']
).ngroup()

In [19]:
print(f'Número de Linhas: {df.shape[0]}\nNúmero de Galpões: {df['IdGalpao'].nunique()}')

Número de Linhas: 43739
Número de Galpões: 522


In [20]:
cidades = ['Rio Branco', 'Cruzeiro do Sul', 'Sena Madureira', 'Tarauacá', 'Feijó',
           'Maceió', 'Arapiraca', 'Palmeira dos Índios', 'Rio Largo', 'Penedo',
           'Macapá', 'Santana', 'Laranjal do Jari', 'Oiapoque', 'Mazagão',
           'Manaus', 'Parintins', 'Itacoatiara', 'Manacapuru', 'Coari',
           'Salvador', 'Feira de Santana', 'Vitória da Conquista', 'Ilhéus', 'Porto Seguro',
           'Fortaleza', 'Juazeiro do Norte', 'Sobral', 'Caucaia', 'Crato',
           'Brasília', 'Taguatinga', 'Ceilândia', 'Samambaia', 'Planaltina',
           'Vitória', 'Vila Velha', 'Serra', 'Cariacica', 'Linhares',
           'Goiânia', 'Anápolis', 'Aparecida de Goiânia', 'Rio Verde', 'Luziânia',
           'São Luís', 'Imperatriz', 'Caxias', 'Timon', 'Codó',
           'Cuiabá', 'Várzea Grande', 'Rondonópolis', 'Sinop', 'Tangará da Serra',
           'Campo Grande', 'Dourados', 'Três Lagoas', 'Corumbá', 'Ponta Porã',
           'Belo Horizonte', 'Uberlândia', 'Contagem', 'Juiz de Fora', 'Ouro Preto',
           'Belém', 'Ananindeua', 'Santarém', 'Marabá', 'Altamira',
           'João Pessoa', 'Campina Grande', 'Santa Rita', 'Patos', 'Bayeux',
           'Curitiba', 'Londrina', 'Maringá', 'Foz do Iguaçu', 'Cascavel',
           'Recife', 'Jaboatão dos Guararapes', 'Olinda', 'Caruaru', 'Petrolina',
           'Teresina', 'Parnaíba', 'Picos', 'Piripiri', 'Floriano',
           'Rio de Janeiro', 'Niterói', 'Campos dos Goytacazes', 'Petrópolis', 'Volta Redonda',
           'Natal', 'Mossoró', 'Parnamirim', 'Caicó', 'Assu',
           'Porto Alegre', 'Caxias do Sul', 'Pelotas', 'Santa Maria', 'Gramado',
           'Porto Velho', 'Ji-Paraná', 'Ariquemes', 'Vilhena', 'Cacoal',
           'Boa Vista', 'Rorainópolis', 'Caracaraí', 'Mucajaí', 'Alto Alegre',
           'Florianópolis', 'Joinville', 'Blumenau', 'Chapecó', 'Balneário Camboriú',
           'São Paulo', 'Campinas', 'Santos', 'São José dos Campos', 'Ribeirão Preto',
           'Aracaju', 'Nossa Senhora do Socorro', 'Lagarto', 'Itabaiana', 'Estância',
           'Palmas', 'Araguaína', 'Gurupi', 'Porto Nacional', 'Paraíso do Tocantins']

In [21]:
df['CdGalpao'] = np.random.choice(cidades)

In [22]:
cidades_por_estado = {
    'AC': ['Rio Branco', 'Cruzeiro do Sul', 'Sena Madureira', 'Tarauacá', 'Feijó'],
    'AL': ['Maceió', 'Arapiraca', 'Palmeira dos Índios', 'Rio Largo', 'Penedo'],
    'AP': ['Macapá', 'Santana', 'Laranjal do Jari', 'Oiapoque', 'Mazagão'],
    'AM': ['Manaus', 'Parintins', 'Itacoatiara', 'Manacapuru', 'Coari'],
    'BA': ['Salvador', 'Feira de Santana', 'Vitória da Conquista', 'Ilhéus', 'Porto Seguro'],
    'CE': ['Fortaleza', 'Juazeiro do Norte', 'Sobral', 'Caucaia', 'Crato'],
    'DF': ['Brasília', 'Taguatinga', 'Ceilândia', 'Samambaia', 'Planaltina'],
    'ES': ['Vitória', 'Vila Velha', 'Serra', 'Cariacica', 'Linhares'],
    'GO': ['Goiânia', 'Anápolis', 'Aparecida de Goiânia', 'Rio Verde', 'Luziânia'],
    'MA': ['São Luís', 'Imperatriz', 'Caxias', 'Timon', 'Codó'],
    'MT': ['Cuiabá', 'Várzea Grande', 'Rondonópolis', 'Sinop', 'Tangará da Serra'],
    'MS': ['Campo Grande', 'Dourados', 'Três Lagoas', 'Corumbá', 'Ponta Porã'],
    'MG': ['Belo Horizonte', 'Uberlândia', 'Contagem', 'Juiz de Fora', 'Ouro Preto'],
    'PA': ['Belém', 'Ananindeua', 'Santarém', 'Marabá', 'Altamira'],
    'PB': ['João Pessoa', 'Campina Grande', 'Santa Rita', 'Patos', 'Bayeux'],
    'PR': ['Curitiba', 'Londrina', 'Maringá', 'Foz do Iguaçu', 'Cascavel'],
    'PE': ['Recife', 'Jaboatão dos Guararapes', 'Olinda', 'Caruaru', 'Petrolina'],
    'PI': ['Teresina', 'Parnaíba', 'Picos', 'Piripiri', 'Floriano'],
    'RJ': ['Rio de Janeiro', 'Niterói', 'Campos dos Goytacazes', 'Petrópolis', 'Volta Redonda'],
    'RN': ['Natal', 'Mossoró', 'Parnamirim', 'Caicó', 'Assu'],
    'RS': ['Porto Alegre', 'Caxias do Sul', 'Pelotas', 'Santa Maria', 'Gramado'],
    'RO': ['Porto Velho', 'Ji-Paraná', 'Ariquemes', 'Vilhena', 'Cacoal'],
    'RR': ['Boa Vista', 'Rorainópolis', 'Caracaraí', 'Mucajaí', 'Alto Alegre'],
    'SC': ['Florianópolis', 'Joinville', 'Blumenau', 'Chapecó', 'Balneário Camboriú'],
    'SP': ['São Paulo', 'Campinas', 'Santos', 'São José dos Campos', 'Ribeirão Preto'],
    'SE': ['Aracaju', 'Nossa Senhora do Socorro', 'Lagarto', 'Itabaiana', 'Estância'],
    'TO': ['Palmas', 'Araguaína', 'Gurupi', 'Porto Nacional', 'Paraíso do Tocantins']
}

In [23]:
cidade_uf_map = {
    cidade: uf
    for uf, cidades in cidades_por_estado.items()
    for cidade in cidades
}

In [24]:
df['UfGalpao'] = df['CdGalpao'].map(cidade_uf_map)

In [25]:
df.columns

Index(['IdPedido', 'IddEntregador', 'NtEntregador', 'LatGalpao', 'LonGalpao',
       'LatDestino', 'LonDestino', 'DtPedido', 'Hrpedido', 'HrColeta', 'Clima',
       'Transito', 'Veiculo', 'Area', 'TpEntrega', 'CtProduto', 'IdEntregador',
       'CtEntregador', 'NmEntregador', 'GrEntregador', 'IdGalpao', 'CdGalpao',
       'UfGalpao'],
      dtype='object')

In [26]:
df_galpoes = df[
    ['IdGalpao', 'CdGalpao', 'UfGalpao', 'LatGalpao', 'LonGalpao']
]

#Tabela de Pedidos

In [27]:
df['CdEntrega'] = np.random.choice(cidades)

In [28]:
df['UfEntrega'] = df['CdEntrega'].map(cidade_uf_map)

In [29]:
df.columns

Index(['IdPedido', 'IddEntregador', 'NtEntregador', 'LatGalpao', 'LonGalpao',
       'LatDestino', 'LonDestino', 'DtPedido', 'Hrpedido', 'HrColeta', 'Clima',
       'Transito', 'Veiculo', 'Area', 'TpEntrega', 'CtProduto', 'IdEntregador',
       'CtEntregador', 'NmEntregador', 'GrEntregador', 'IdGalpao', 'CdGalpao',
       'UfGalpao', 'CdEntrega', 'UfEntrega'],
      dtype='object')

In [30]:
df_entregas = df[
    ['IdPedido', 'LatDestino', 'LonDestino', 'DtPedido', 'Hrpedido', 'HrColeta', 'Clima',
    'Transito', 'Area', 'TpEntrega', 'CtProduto', 'IdEntregador', 'IdGalpao', 'CdEntrega',
    'UfEntrega']
]

# Subindo as Tabelas no Banco

In [31]:
conn = sqlite3.connect('meu_banco.db')

In [32]:
df_entregadores.to_sql('logiPrime.Entregadores', conn, if_exists='replace', index=False)
df_galpoes.to_sql('logiPrime.Galpoes', conn, if_exists='replace', index=False)
df_entregas.to_sql('logiPrime.Entregas', conn, if_exists='replace', index=False)

43739

In [33]:
from google.colab import files
files.download('meu_banco.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>